In [1]:
import pandas as pd  

# Load datasets
orders = pd.read_csv("MP-1/Orders_SwiftShip_Logistics.csv")  
carriers = pd.read_csv("MP-1/Carriers_SwiftShip_Logistics.csv")  
customer_feedback = pd.read_csv("MP-1/Customer_Feedback_SwiftShip_Logistics.csv")  
delivery_costs = pd.read_csv("MP-1/Delivery_Costs_SwiftShip_Logistics.csv")  


In [2]:
orders.isna().sum()


Order_ID           0
Order_Date         0
Delivery_Date      0
Delivery_Status    0
City               0
Carrier_ID         0
Delivery_Zone      0
dtype: int64

In [3]:
orders.duplicated().sum()

0

In [4]:
orders["Order_Date"] = pd.to_datetime(orders["Order_Date"], errors="coerce")
orders["Delivery_Date"] = pd.to_datetime(orders["Delivery_Date"], errors="coerce")

In [5]:
orders.dtypes

Order_ID                   object
Order_Date         datetime64[ns]
Delivery_Date      datetime64[ns]
Delivery_Status            object
City                       object
Carrier_ID                 object
Delivery_Zone              object
dtype: object

In [6]:
carriers.duplicated().sum()

0

In [7]:
carriers.isna().sum()

Carrier_ID           0
Carrier_Name         0
Avg_Delivery_Cost    0
Defect_Rate          0
dtype: int64

In [8]:
delivery_costs.duplicated().sum()

0

In [9]:
customer_feedback.isna().sum()

Order_ID                         0
Delivery_Experience_Rating       0
Issue_Reported                5770
dtype: int64

In [10]:
customer_feedback["Issue_Reported"] = customer_feedback["Issue_Reported"].fillna("None")

In [11]:
customer_feedback.isna().sum()

Order_ID                      0
Delivery_Experience_Rating    0
Issue_Reported                0
dtype: int64

In [12]:
customer_feedback["Issue_Reported"] = customer_feedback["Issue_Reported"].astype("category")


In [13]:
orders['Delivery_Status'] = orders['Delivery_Status'].astype("category")

In [14]:
orders["Delivery_Zone"] = orders["Delivery_Zone"].astype("category")


In [15]:
orders["City"] = orders["City"].astype("category")


In [16]:
carriers["Carrier_Name"] = carriers["Carrier_Name"].astype("category")

In [17]:
df = pd.merge(orders, delivery_costs, on="Order_ID", how="left")

In [18]:
df = pd.merge(df, customer_feedback, on="Order_ID", how="left")

In [19]:
df = pd.merge(df, carriers, on="Carrier_ID", how="left")

In [20]:
df.head()

,Order_ID,Order_Date,Delivery_Date,Delivery_Status,City,Carrier_ID,Delivery_Zone,Distance_km,Fuel_Charge,Total_Delivery_Cost,Delivery_Experience_Rating,Issue_Reported,Carrier_Name,Avg_Delivery_Cost,Defect_Rate
0,01603720,2023-06-13,2023-06-17,Delivered,Pune,C006,Central,44.68,84.49,96.85,4,None,Carrier_6,82.09,0.04
1,8d1d2ed2,2023-01-29,2023-02-02,Cancelled,Ahmedabad,C004,East,44.09,83.38,131.44,1,Wrong Address,Carrier_4,93.62,0.04
2,132104b2,2023-01-07,2023-01-09,Delayed,Hyderabad,C004,East,29.46,55.71,88.99,5,Damaged Package,Carrier_4,93.62,0.04
3,ff04d8fb,2023-03-12,2023-03-14,Delivered,Delhi,C007,North,23.73,44.88,83.54,2,Wrong Address,Carrier_7,79.32,0.03
4,7efd5d10,2023-03-04,2023-03-09,Delivered,Chennai,C003,North,21.23,40.15,68.90,1,Late Delivery,Carrier_3,84.25,0.06


In [21]:
# df.to_csv("Merged_SwiftShip_Logistics.csv", index=False)

## KPIs

#### On-Time Delivery Rate

In [22]:
on_time_deliveries = df[df["Delivery_Status"].str.lower().str.contains("delivered")].shape[0]
total_orders = df.shape[0]
on_time_delivery_rate = (on_time_deliveries / total_orders) * 100
print(f"On-Time Delivery Rate: {on_time_delivery_rate:.2f}%")

On-Time Delivery Rate: 69.19%


#### Avg. Delivery Time

In [23]:
df['Delivery_Days'] = (df['Delivery_Date'] - df['Order_Date']).dt.days

In [24]:
avg_delivery_time = df["Delivery_Days"].mean()
print(f"Average Delivery Time: {avg_delivery_time:.2f} days")

Average Delivery Time: 2.98 days


#### Cost per Delivery

In [25]:
avg_cost_per_delivery = df["Total_Delivery_Cost"].mean()
print(f"Average Cost per Delivery: ₹{avg_cost_per_delivery:.2f}")

Average Cost per Delivery: ₹79.70


## Group-By Analysis

#### City-Wise Performance Analysis

In [26]:
df["On_Time_Delivery"] = df["Delivery_Status"].str.lower().str.contains("delivered").astype(int)

In [27]:
city_performance = df.groupby("City").agg(
    On_Time_Delivery_Rate=("On_Time_Delivery", "mean"),
    Avg_Delivery_Time=("Delivery_Days", "mean"),
    Avg_Cost_Per_Delivery=("Total_Delivery_Cost", "mean")
).reset_index()
# In percentage
city_performance["On_Time_Delivery_Rate"] *= 100  
print(city_performance)


        City  On_Time_Delivery_Rate  Avg_Delivery_Time  Avg_Cost_Per_Delivery
0  Ahmedabad              70.344288           3.025039              80.456901
1  Bangalore              67.058824           2.957143              77.931555
2    Chennai              68.510985           2.969081              80.692343
3      Delhi              68.813826           2.970149              80.653488
4  Hyderabad              67.836735           2.972245              78.282612
5    Kolkata              70.975416           3.021412              80.142189
6     Mumbai              70.040816           2.941224              79.284457
7       Pune              69.749810           2.967400              79.994094


C:\Users\Ria\AppData\Local\Temp\ipykernel_13824\792620082.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  city_performance = df.groupby("City").agg(


#### Carrier Performance Analysis

In [28]:
carrier_performance = df.groupby("Carrier_Name").agg(
    Avg_Defect_Rate=("Defect_Rate", "mean"),
    Avg_Delivery_Cost=("Total_Delivery_Cost", "mean"),
    On_Time_Delivery_Rate=("On_Time_Delivery", "mean")
).reset_index()
carrier_performance["Avg_Defect_Rate"] *= 100
carrier_performance["On_Time_Delivery_Rate"] *= 100
print(carrier_performance)

  Carrier_Name  Avg_Defect_Rate  Avg_Delivery_Cost  On_Time_Delivery_Rate
0    Carrier_1              7.0          78.916507              69.469469
1   Carrier_10              8.0          79.566834              70.184426
2    Carrier_2              3.0          79.276073              71.198389
3    Carrier_3              6.0          79.987049              67.472306
4    Carrier_4              4.0          79.026577              68.653846
5    Carrier_5              9.0          81.367308              65.976331
6    Carrier_6              4.0          79.913982              69.560878
7    Carrier_7              3.0          79.878684              69.941061
8    Carrier_8              6.0          79.393617              69.334719
9    Carrier_9              2.0          79.653300              70.189432


C:\Users\Ria\AppData\Local\Temp\ipykernel_13824\4248587591.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  carrier_performance = df.groupby("Carrier_Name").agg(


#### Delivery Zone Performance Analysis

In [29]:
zone_performance = df.groupby("Delivery_Zone").agg(
    Avg_Delivery_Time=("Delivery_Days", "mean"),
    On_Time_Delivery_Rate=("On_Time_Delivery", "mean"),
    Avg_Cost_Per_Delivery=("Total_Delivery_Cost", "mean")
).reset_index()
city_performance["On_Time_Delivery_Rate"] *= 100  
print(zone_performance)

  Delivery_Zone  Avg_Delivery_Time  On_Time_Delivery_Rate  \
0       Central           2.974902               0.699803   
1          East           2.989232               0.691140   
2         North           2.930556               0.691358   
3         South           3.006513               0.690381   
4          West           2.988917               0.686650   

   Avg_Cost_Per_Delivery  
0              79.359291  
1              78.812942  
2              79.849239  
3              81.348898  
4              79.155758  


C:\Users\Ria\AppData\Local\Temp\ipykernel_13824\1621418457.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  zone_performance = df.groupby("Delivery_Zone").agg(


# Summary Insights:

- On-Time Delivery Rate is 69.19%
- Average Delivery Time is 2.98 days
- Average Cost per Delivery is ₹79.70

### Carrier Performance Analysis
**Best Carrier for On-Time Delivery**:
- Carrier_2 (71.20%) has the highest on-time delivery rate, followed closely by Carrier_9 (70.19%) and Carrier_10 (70.18%).

**Most Cost-Effective Carrier**:

- Carrier_1 has the lowest average delivery cost (₹78.92), making it more cost-efficient.

**Worst Performing Carrier**:

- Carrier_5 has the highest defect rate (9%) and the lowest on-time delivery rate (65.98%).

- Carrier_3 also struggles, with low on-time delivery (67.47%) and a high defect rate (6%).

### Delivery Zone Performance Analysis
**Fastest Zone for Delivery:**
- The North zone has the lowest Avg. Delivery Time (2.93 days), making it the most efficient.

**Most Expensive Zone:**

- The South zone has the highest Avg. Cost per Delivery (₹81.35), possibly due to longer distances or higher fuel costs.

**Delivery Zone with Best On-Time Performance:**

- The Central zone has the highest on-time delivery rate (69.98%), but overall, all zones are below 70%, indicating room for improvement.

### City-Wise Performance Analysis
**Best City for On-Time Delivery:**

- Kolkata (70.98%) has the highest on-time delivery rate, followed by Ahmedabad (70.34%).

- Worst city: Bangalore (67.06%) struggles with timely deliveries.

**Most Cost-Effective City:**

- Bangalore has the lowest Avg. Cost per Delivery (₹77.93), making it the most efficient in terms of cost.

**City with Slowest Deliveries:**

- Ahmedabad has the longest Avg. Delivery Time (3.03 days), followed by Kolkata (3.02 days).